# TECH CHALLENGE VIDEO RECOGNITION

1.0 - Etapa Montagem do google Drive e Captura do video

In [2]:
# Video Analysis Tech Challenge - Google Colab Notebook
# Etapa 1: Instalação de dependências + Conectar com o Google Drive
from google.colab import drive
import os
import cv2

drive.mount('/content/drive')

# Caminho do vídeo dentro do seu Google Drive
drive_video_path = '/content/drive/MyDrive/VideoAnaliseTechChallenge.mp4'  # <-- Substitua pelo caminho correto
VIDEO_PATH = drive_video_path

os.makedirs("output", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1.1 - Etapa Instalação de Dependencias

In [ ]:
!pip uninstall -y opencv-python opencv-python-headless mediapipe deepface protobuf
!pip install mediapipe==0.10.21 deepface==0.0.93 opencv-python==4.9.0.80 protobuf==4.25.3

2.0 - Metodo para reconhecimento facial com DeepFace

In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # força CPU

from deepface import DeepFace
import cv2

def detect_faces_and_emotions(frame):
    frame_resized = cv2.resize(frame, (640, 360)) #Video Recortado para acelera

    try:
        results = DeepFace.analyze(
            frame_resized,
            actions=["emotion"],
            detector_backend="mediapipe",  # mais leve
            enforce_detection=True  # só conta se realmente detectar rosto
        )

        if isinstance(results, dict):
            results = [results]

        output = []
        for r in results:
            if r['region']['w'] > 0 and r['region']['h'] > 0:
                box = (
                    r['region']['x'],
                    r['region']['y'],
                    r['region']['w'],
                    r['region']['h']
                )
                emotion = r['dominant_emotion']
                output.append({'box': box, 'emotion': emotion})

        return output

    except Exception as e:
        # Apenas loga, não conta falso positivo
        # print(f"[ERRO] Falha ao analisar frame: {e}")
        return []

2.1 - Etapa Reconhecimento facial por frames e analise de emoções

In [8]:
import cv2
from tqdm import tqdm
from collections import Counter

cap = cv2.VideoCapture(VIDEO_PATH)

# Calcula número de frames e FPS
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
duracao_seg = total_frames / fps

print(f"Total de frames: {total_frames}")
print(f"FPS: {fps}")
print(f"Duração: {duracao_seg:.2f} segundos ({duracao_seg/60:.2f} minutos)")

# Define step para pegar pelo menos 70% dos frames
frames_alvo = int(total_frames * 0.7)
step = max(1, total_frames // frames_alvo)  # garante que não seja 0

# Contadores
total_rostos = 0
total_emocoes = 0
frame_count = 0
frames_processados = 0

# Contador para emoções
emocao_counter = Counter()

# Barra de progresso
pbar = tqdm(total=total_frames, desc="Processando vídeo", unit="frame")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % step == 0:  # processa só os frames no intervalo
        detections = detect_faces_and_emotions(frame)
        if detections:  # só conta se detectar
            total_rostos += len(detections)
            total_emocoes += len(detections)
            for det in detections:
                emocao_counter[det['emotion']] += 1
        frames_processados += 1

    frame_count += 1
    pbar.update(1)

cap.release()
pbar.close()

print(f"\nFrames processados: {frames_processados} ({(frames_processados/total_frames)*100:.2f}%)")
print(f"Total de rostos detectados: {total_rostos}")
print(f"Total de emoções detectadas: {total_emocoes}")

print("\nRanking de Emoções:")
for emocao, qtd in emocao_counter.most_common():
    print(f"{emocao}: {qtd}")


Total de frames: 3326
FPS: 30.0
Duração: 110.87 segundos (1.85 minutos)


Processando vídeo: 100%|██████████| 3326/3326 [00:59<00:00, 56.02frame/s] 


Frames processados: 3326 (100.00%)
Total de rostos detectados: 176
Total de emoções detectadas: 176

Ranking de Emoções:
happy: 71
fear: 45
sad: 30
angry: 16
surprise: 12
neutral: 2


Etapa 3.0 - Analise de Atividades com MediaPipe Pose e Keypoints

In [24]:
import cv2
import mediapipe as mp
from tqdm import tqdm
from collections import Counter, deque
import math

# ---------------- Funções utilitárias ----------------

def distancia(p1, p2):
    return math.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)

def normaliza_distancia(dist, referencia):
    return dist / referencia if referencia > 0 else 0

# ---------------- Função de classificação ----------------

def classifica_atividade(landmarks, historico_movimentos):
    left_shoulder = landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
    right_shoulder = landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
    distancia_ombros = distancia(left_shoulder, right_shoulder)

    left_wrist = landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]
    right_wrist = landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value]
    dist_punhos = normaliza_distancia(distancia(left_wrist, right_wrist), distancia_ombros)
    punhos_baixos = (left_wrist.y > left_shoulder.y) and (right_wrist.y > right_shoulder.y)

    # Conversando
    nose = landmarks[mp_pose.PoseLandmark.NOSE.value]
    left_ear = landmarks[mp_pose.PoseLandmark.LEFT_EAR.value]
    right_ear = landmarks[mp_pose.PoseLandmark.RIGHT_EAR.value]
    dist_orelhas = normaliza_distancia(distancia(left_ear, right_ear), distancia_ombros)
    inclinacao_cabeca = abs(left_ear.y - right_ear.y)

    # Quadris e joelhos para postura
    left_hip = landmarks[mp_pose.PoseLandmark.LEFT_HIP.value]
    right_hip = landmarks[mp_pose.PoseLandmark.RIGHT_HIP.value]
    left_knee = landmarks[mp_pose.PoseLandmark.LEFT_KNEE.value]
    right_knee = landmarks[mp_pose.PoseLandmark.RIGHT_KNEE.value]

    centro_quadril_x = (left_hip.x + right_hip.x) / 2

    # Atualiza histórico
    historico_movimentos.append({
        'ombro': distancia_ombros,
        'punhos': dist_punhos,
        'quadris': centro_quadril_x
    })

    # Heurística: Sentado (joelho mais alto que quadril)
    if (left_knee.y < left_hip.y) and (right_knee.y < right_hip.y):
        return "Sentado"

    # Heurística: Aperto de Mão
    if dist_punhos < 0.45 and punhos_baixos:
        return "Aperto de Mão"

    # Heurística: Conversando
    if (0.26 < dist_orelhas < 0.50 and inclinacao_cabeca > 0.015):
        return "Conversando"

    # Heurística: Dançando
    if len(historico_movimentos) >= 5:
        ombro_vals = [h['ombro'] for h in historico_movimentos if h['ombro'] is not None]
        quadril_vals = [h['quadris'] for h in historico_movimentos if h['quadris'] is not None]

        if len(ombro_vals) >= 2 and len(quadril_vals) >= 2:
            variacao_ombros = max(ombro_vals) - min(ombro_vals)
            variacao_quadris = max(quadril_vals) - min(quadril_vals)
            variacao_total = (variacao_quadris + variacao_ombros) / 2

            if variacao_total > 0.05:
                return "Dançando"

    # Se a pose for detectada, mas não encaixa: Pessoa em Pé
    return "Pessoa em Pé"

# ---------------- Função principal ----------------

def analisar_video(video_path):
    global mp_pose
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    atividades_detectadas = []
    historico_movimentos = deque(maxlen=10)

    for _ in tqdm(range(total_frames), desc="Analisando frames"):
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb_frame)

        if results.pose_landmarks:
            atividade = classifica_atividade(results.pose_landmarks.landmark, historico_movimentos)
        else:
            atividade = "Anomalia"

        atividades_detectadas.append(atividade)

    cap.release()
    pose.close()

    total_analisados = len(atividades_detectadas)
    contagem_atividades = Counter(atividades_detectadas)

    print(f"\n📊 Relatório Final - Análise de Atividades")
    print(f"Frames analisados: {total_analisados}")
    print(f"Quantidade de Atividades Diferentes Registradas: {len(contagem_atividades)}")
    print("Ranking de Atividades:")
    for atividade, quantidade in contagem_atividades.most_common():
        print(f"{atividade}: {quantidade}")

# ---------------- Execução ----------------

if __name__ == "__main__":
    VIDEO_PATH = "/content/drive/MyDrive/VideoAnaliseTechChallenge.mp4"
    analisar_video(VIDEO_PATH)


Analisando frames: 100%|██████████| 3326/3326 [02:32<00:00, 21.87it/s]


📊 Relatório Final - Análise de Atividades
Frames analisados: 3326
Quantidade de Atividades Diferentes Registradas: 6
Ranking de Atividades:
Pessoa em Pé: 1319
Conversando: 643
Anomalia: 517
Sentado: 454
Dançando: 316
Aperto de Mão: 77
